# MFVI predictive-variance — full reproducibility gate

Use a **CPU** Colab runtime. Upload the supplied input archive when prompted, then run each cell in order. The notebook runs the complete nine-dataset author path, an independent audit, and the fail-closed publication gate.

In [ ]:
from google.colab import files
from pathlib import Path
import os
import subprocess
import sys

uploaded = files.upload()
archive = next((Path(name) for name in uploaded if name.endswith('.tar.gz')), None)
if archive is None:
    raise RuntimeError('Upload RG7maF4bGu-colab-bundle.tar.gz')
subprocess.run(['tar', '-tzf', str(archive)], check=True, stdout=subprocess.DEVNULL)
subprocess.run(['tar', '-xzf', str(archive), '-C', '/content'], check=True)
PROJECT = Path('/content/icml26-repro-RG7maF4bGu-mfvi-predictive-variance')
if not PROJECT.is_dir():
    raise RuntimeError(f'Expected project directory missing: {PROJECT}')
os.chdir(PROJECT)
print('Input bundle extracted to:', PROJECT)

In [ ]:
# Colab's Python image can omit ensurepip, so do not create a virtual environment.
# The gate expects .venv/bin/python; make that path a link to this exact Colab interpreter.
venv_python = sys.executable
shim_dir = PROJECT / '.venv/bin'
shim_dir.mkdir(parents=True, exist_ok=True)
shim_python = shim_dir / 'python'
if shim_python.exists() or shim_python.is_symlink():
    shim_python.unlink()
shim_python.symlink_to(venv_python)
subprocess.run([venv_python, '-m', 'pip', 'install', '--quiet', '--upgrade', 'pip'], check=True)
subprocess.run([venv_python, '-m', 'pip', 'install', '--quiet', '-r', 'repro/requirements.txt'], check=True)
torch_check = subprocess.run([venv_python, '-c', 'import torch; print(torch.__version__)'])
if torch_check.returncode != 0:
    subprocess.run([venv_python, '-m', 'pip', 'install', '--quiet', '--index-url', 'https://download.pytorch.org/whl/cpu', 'torch>=2.5,<2.7'], check=True)
subprocess.run([venv_python, '-c', 'import torch, pandas, numpy, scipy; print({\"torch\": torch.__version__, \"cuda\": torch.cuda.is_available()})'], check=True)

In [ ]:
# The tarball preserves the pinned checkout's Git metadata; authorize only this extracted directory.
subprocess.run(['git', 'config', '--global', '--add', 'safe.directory', str(PROJECT / 'upstream')], check=True)
# Preflight validates all pins and controls; the next command is the complete full-scale gate.
environment = {**os.environ, 'CUDA_VISIBLE_DEVICES': '', 'OPENBLAS_NUM_THREADS': '1', 'OMP_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1'}
subprocess.run([venv_python, 'repro/src/verify_mfvi.py', '--mode', 'synthetic', '--output', 'outputs/colab_synthetic_preflight.json'], check=True, env=environment)
subprocess.run(['bash', 'repro/src/run_full_gate.sh'], check=True, env=environment)
subprocess.run([venv_python, '-m', 'pytest', '-q', 'repro/tests'], check=True, env=environment)
print((PROJECT / 'outputs/prepublish_gate.json').read_text())

In [ ]:
# Download the outputs-only evidence archive and upload that archive back to this chat.
result_archive = Path('/content/RG7maF4bGu-colab-results.tar.gz')
subprocess.run(['tar', '-czf', str(result_archive), '-C', str(PROJECT), 'outputs'], check=True)
subprocess.run(['sha256sum', str(result_archive)], check=True)
files.download(str(result_archive))